# Clase 11 — Balancea tus datos y entrena tu modelo LSTM final

**Manual asociado:** [manual/11_caso_cripto_modelo_final.md](../manual/11_caso_cripto_modelo_final.md)

## Resumen para la clase

- Balanceo manual: separa por clase, recorta al tamaño del grupo más pequeño, combina y mezcla.
- Arquitectura final: LSTM apiladas + Dropout + BatchNormalization.
- `CuDNNLSTM` ya no existe en TensorFlow 2.x: usa `LSTM` directamente.
- Corrige siempre los argumentos de `ModelCheckpoint`: deben ir al callback, no perderse en un `str.format()`.

In [ ]:
import pickle

with open("secuencias_entrenamiento.pickle", "rb") as f:
    secuencias_entrenamiento = pickle.load(f)
with open("secuencias_validacion.pickle", "rb") as f:
    secuencias_validacion = pickle.load(f)

In [ ]:
import random


def balancea_secuencias(datos_secuenciales):
    compras, ventas = [], []
    for secuencia, target in datos_secuenciales:
        (ventas if target == 0 else compras).append([secuencia, target])

    random.shuffle(compras)
    random.shuffle(ventas)

    limite = min(len(compras), len(ventas))
    datos_balanceados = compras[:limite] + ventas[:limite]
    random.shuffle(datos_balanceados)
    return datos_balanceados


secuencias_entrenamiento = balancea_secuencias(secuencias_entrenamiento)
secuencias_validacion = balancea_secuencias(secuencias_validacion)

print(f"Entrenamiento balanceado: {len(secuencias_entrenamiento)} secuencias")
print(f"Validacion balanceada: {len(secuencias_validacion)} secuencias")

In [ ]:
import numpy as np


def separa_features_y_target(datos_secuenciales):
    X, y = [], []
    for secuencia, target in datos_secuenciales:
        X.append(secuencia)
        y.append(target)
    return np.array(X), np.array(y)


train_x, train_y = separa_features_y_target(secuencias_entrenamiento)
validation_x, validation_y = separa_features_y_target(secuencias_validacion)
print("Forma de train_x:", train_x.shape)

In [ ]:
import time
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, LSTM, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import TensorBoard, ModelCheckpoint, EarlyStopping

EPOCHS = 10
BATCH_SIZE = 64
NOMBRE_MODELO = f"LSTM-cripto-{int(time.time())}"

model = Sequential([
    LSTM(128, input_shape=(train_x.shape[1:]), return_sequences=True),
    Dropout(0.2),
    BatchNormalization(),

    LSTM(128, return_sequences=True),
    Dropout(0.1),
    BatchNormalization(),

    LSTM(128),
    Dropout(0.2),
    BatchNormalization(),

    Dense(32, activation="relu"),
    Dropout(0.2),

    Dense(2, activation="softmax"),
])

optimizador = Adam(learning_rate=0.001, decay=1e-6)
model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizador, metrics=["accuracy"])
model.summary()

In [ ]:
import os

os.makedirs("modelos", exist_ok=True)

tensorboard_callback = TensorBoard(log_dir=f"logs/{NOMBRE_MODELO}")

checkpoint_callback = ModelCheckpoint(
    filepath="modelos/epoch-{epoch:02d}-val_acc-{val_accuracy:.3f}.keras",
    monitor="val_accuracy",
    verbose=1,
    save_best_only=True,
    mode="max",
)

early_stopping_callback = EarlyStopping(monitor="val_loss", patience=3)

In [ ]:
model.fit(
    train_x, train_y,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(validation_x, validation_y),
    callbacks=[tensorboard_callback, checkpoint_callback, early_stopping_callback],
)

In [ ]:
loss, accuracy = model.evaluate(validation_x, validation_y)
print(f"Perdida de validacion: {loss:.4f}")
print(f"Precision de validacion: {accuracy:.4f}")

## Cierre de la clase

- Recuerda: este modelo es una herramienta educativa, no un consejo de inversión.
- En la clase 12 cierras el curso repasando este caso junto a MNIST y gatos vs. perros como casos de uso reales.